# PE-AV Embedding Extraction

Extract audio, video, and joint (AV) embeddings from PE-AV model variants.
Configure `MODEL_IDS` below to run one or multiple variants.

**Environment:** `avtransformer`  
**Output:** `/outputs/model_embeddings/{model_name}/{bin_sec}s/{model_name}_{v,a,av}.npy`

In [12]:
import gc
import os
import re
from pathlib import Path

import numpy as np
import torch
from natsort import natsorted
from transformers import AutoModel, AutoProcessor

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
# Add or remove entries to control which model variants are processed.
# hf_id:      HuggingFace model identifier used for from_pretrained()
# modalities: which output keys to save — subset of ["v", "a", "av"]
MODEL_CONFIGS = {
    "pe-av-small-16-frame": {
        "hf_id": "facebook/pe-av-small-16-frame",
        "modalities": ["v", "a", "av"],
    },
    # "pe-av-small": {
    #     "hf_id": "facebook/pe-av-small",
    #     "modalities": ["v", "a", "av"],
    # },
    # "pe-av-base": {
    #     "hf_id": "facebook/pe-av-base",
    #     "modalities": ["v", "a", "av"],
    # },
    # "pe-av-base-16-frame": {
    #     "hf_id": "facebook/pe-av-base-16-frame",
    #     "modalities": ["v", "a", "av"],
    # },
    # "pe-av-large": {
    #     "hf_id": "facebook/pe-av-large",
    #     "modalities": ["v", "a", "av"],
    # },
    # "pe-av-large-16-frame": {
    #     "hf_id": "facebook/pe-av-large-16-frame",
    #     "modalities": ["v", "a", "av"],
    # },
}

SEGMENT_DURATIONS = [5.0, 10.0]  # seconds
BATCH_SIZE = 8
LOCAL_FILES_ONLY = True

DATA_BASE   = Path("/home/amin/Research/Representation/Movie/data/segmented_stimulus/filtered")
RUNS_BASE = Path("/home/amin/Research/Representation/Movie/data/runs")
OUTPUT_BASE = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings")
# ─────────────────────────────────────────────────────────────────────────────

## Helper Functions

In [19]:
def get_model_shortname(model_id: str) -> str:
    """Convert HuggingFace model ID to a short filesystem-safe name."""
    return model_id.split("/")[-1]


def find_chunk_pairs(data_base: Path, seg_sec: float) -> list[tuple[Path, Path]]:
    '''Return sorted (video_mp4, audio_wav) pairs for a given segment duration.'''
    pairs = []
    for vid_subdir in natsorted(data_base.glob("Video*")):
        m = re.search(r"Video(\d+)$", vid_subdir.name)
        if not m:
            continue
        idx = m.group(1)
        chunk_dir = vid_subdir / f"Video{idx}_chunks_{int(seg_sec)}s"
        audio_dir = data_base / f"Audio{idx}" / f"Audio{idx}_chunks_{int(seg_sec)}s"
        if not chunk_dir.exists() or not audio_dir.exists():
            continue
        for vid in natsorted(chunk_dir.glob("*_part_*.mp4")):
            part_num = re.search(r"_part_(\d+)", vid.stem).group(1)
            aud = audio_dir / f"Audio{idx}_part_{part_num}.wav"
            if aud.exists():
                pairs.append((vid, aud))
    return pairs



@torch.no_grad()
def extract_embeddings(model, processor, pairs: list, batch_size: int,
                        device: str) -> dict[str, np.ndarray]:
    """Extract audio, video, and joint embeddings from video/audio file pairs.

    Returns dict with keys 'v', 'a', 'av' each containing (N, D) arrays.
    """
    video_embeds, audio_embeds, av_embeds = [], [], []

    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i + batch_size]
        vid_paths = [str(p[0]) for p in batch]
        aud_paths = [str(p[1]) for p in batch]

        inputs = processor(videos=vid_paths, audio=aud_paths, return_tensors="pt")
        inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                  for k, v in inputs.items()}

        outputs = model(**inputs)
        video_embeds.append(outputs.video_embeds.cpu().numpy())
        audio_embeds.append(outputs.audio_embeds.cpu().numpy())
        av_embeds.append(outputs.audio_video_embeds.cpu().numpy())

        if (i // batch_size) % 10 == 0:
            print(f"    {i + len(batch)}/{len(pairs)} chunks processed", end="\r")

    print()
    return {
        "v":  np.concatenate(video_embeds, axis=0),
        "a":  np.concatenate(audio_embeds, axis=0),
        "av": np.concatenate(av_embeds, axis=0),
    }

## Data Compatibility Check

Verify paired video/audio chunks exist for each segment duration.

In [15]:
print("Chunk pair counts per segment duration:")
for seg_sec in SEGMENT_DURATIONS:
    pairs = find_chunk_pairs(DATA_BASE, seg_sec)
    print(f"  {seg_sec:4.1f}s → {len(pairs):4d} pairs")

Chunk pair counts per segment duration:
   5.0s →  626 pairs
  10.0s →  311 pairs


## Embedding Extraction

Loop over all models in `MODEL_IDS`. Each model is loaded, all segment durations
are processed, and the model is deleted to free GPU memory before the next iteration.

In [ ]:
for model_name, cfg in MODEL_CONFIGS.items():
    model_id = cfg["hf_id"]
    modalities = cfg["modalities"]
    print(f"\n{'='*60}")
    print(f"Model: {model_id}")
    print(f"{'='*60}")

    # Load model
    processor = AutoProcessor.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY)
    model = AutoModel.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY).to(DEVICE)
    model.eval()

    for seg_sec in SEGMENT_DURATIONS:
        out_dir = OUTPUT_BASE / model_name / f"{int(seg_sec)}s"
        out_dir.mkdir(parents=True, exist_ok=True)

        # Skip if all modality files already exist
        if all((out_dir / f"{model_name}_{m}.npy").exists() for m in modalities):
            print(f"  {seg_sec}s — already done, skipping")
            continue

        print(f"  {seg_sec}s — collecting pairs ...")
        pairs = find_chunk_pairs(DATA_BASE, seg_sec)
        print(f"  {seg_sec}s — extracting from {len(pairs)} pairs ...")

        embeds = extract_embeddings(model, processor, pairs, BATCH_SIZE, DEVICE)

        for mod, arr in embeds.items():
            if mod not in modalities:
                continue
            out_path = out_dir / f"{model_name}_{mod}.npy"
            np.save(out_path, arr)
            print(f"    saved {out_path.name}  shape={arr.shape}")

    # Release GPU memory before loading next model
    del model, processor
    torch.cuda.empty_cache()
    gc.collect()
    print(f"  {model_name} complete, GPU memory released")

print("\nAll models done.")

## Full Movie Runs (Optional)

Extract embeddings from full-length movie runs (not pre-chunked segments).
Outputs are saved under `{model}/{bin_sec}s/full/`.

In [ ]:
EXTRACT_FULL_RUNS = False  # Set to True to run this cell

if EXTRACT_FULL_RUNS:
    for model_id in MODEL_IDS:
        short = get_model_shortname(model_id)
        processor = AutoProcessor.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY)
        model = AutoModel.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY).to(DEVICE)
        model.eval()

        for seg_sec in SEGMENT_DURATIONS:
            run_files = natsorted(RUNS_BASE.glob("*.mp4"))
            pairs = [(f, f.with_suffix(".wav")) for f in run_files
                     if f.with_suffix(".wav").exists()]

            out_dir = OUTPUT_BASE / short / f"{int(seg_sec)}s" / "full"
            out_dir.mkdir(parents=True, exist_ok=True)

            embeds = extract_embeddings(model, processor, pairs, BATCH_SIZE, DEVICE)
            for mod, arr in embeds.items():
                np.save(out_dir / f"{short}_{mod}.npy", arr)

        del model, processor
        torch.cuda.empty_cache()
        gc.collect()

## Text-Conditioned Embeddings (AVT)

Feed `text + audio + video` together through PE-AV to obtain three text-grounded
representations per segment:

| suffix | tensor | description |
|--------|--------|-------------|
| `_avt` | `text_audio_video_embeds` | text CLS in AV contrastive space |
| `_at`  | `audio_plus_text_embeds`  | concat(audio\_pooler, text\_CLS) -> head |
| `_vt`  | `video_plus_text_embeds`  | concat(video\_pooler, text\_CLS) -> head |

**Prerequisite:** run `text.ipynb` (Whisper + InternVL3-8B + AudioFlamingo3-HF) then
`rewrite_text_inputs.py` to generate `OUTPUT_BASE/text/{seg}s/rewritten_text_inputs.json`.

In [20]:
import json

TEXT_BASE       = OUTPUT_BASE / "text"
TEXT_MODALITIES = ["avt", "at", "vt", "t"]


@torch.no_grad()
def extract_avt_embeddings(model, processor, pairs, text_inputs, batch_size, device):
    '''Extract text-conditioned AV embeddings; adaptive batch size on CUDA OOM.

    Returns a dict with four keys, all shape (N, D):

    avt  (text_audio_video_embeds)
        text CLS  -->  text_audio_video_head  --> shared AV contrastive space.
        The text-side counterpart of cls_av; trained so that
        audio_video_embeds @ avt.T maximises alignment.

    at  (audio_plus_text_embeds)
        concat(audio_pooler_output, text_CLS)  -->  audio_plus_text_head.
        An explicit audio+text fusion: the clip's transcript/caption is
        concatenated with the audio encoder output *at inference time* before
        being projected through a dedicated head.  Unlike cls_a (which is
        audio-video aligned but text-free at inference), at is conditioned on
        this specific clip's language content.

    vt  (video_plus_text_embeds)
        concat(video_pooler_output, text_CLS)  -->  video_plus_text_head.
        Same idea as 'at' but for the video encoder.

    t   (raw ModernBERT text CLS)
        text_outputs.hidden_states[-1][:, 0]

        hidden_states[-1]  = last transformer layer output, shape (B, seq_len, D).
        [:, 0]             = position 0, which is the [CLS] token.
        ModernBERT (like BERT) always prepends [CLS] at position 0, so index 0
        IS the CLS token -- not just an arbitrary first token.

        Crucially, this is the RAW hidden state BEFORE any projection head.
        Standard BERT pooler_output applies tanh(linear(CLS)) on top; PE-AV
        skips that and uses hidden_states[-1][:, 0] directly as the upstream
        input to all three text heads (avt, at, vt).  So 't' is exactly the
        shared text feature from which avt/at/vt are all derived.

        PE-AV forces output_hidden_states=True internally (modeling line 892)
        before calling the text model, so hidden_states is always populated
        during the trimodal forward pass.
    '''
    avt_list, at_list, vt_list, t_list = [], [], [], []
    cur_size = batch_size
    i = 0
    while i < len(pairs):
        batch      = pairs[i : i + cur_size]
        batch_text = text_inputs[i : i + cur_size]
        vid_paths  = [str(p[0]) for p in batch]
        aud_paths  = [str(p[1]) for p in batch]
        try:
            text_enc = processor.tokenizer(
                batch_text,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=128,
            ).to(device)

            av_inputs = processor(videos=vid_paths, audio=aud_paths, return_tensors="pt")
            av_inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                         for k, v in av_inputs.items()}

            outputs = model(
                input_ids=text_enc.input_ids,
                attention_mask=text_enc.attention_mask,
                pixel_values_videos=av_inputs["pixel_values_videos"],
                input_values=av_inputs["input_values"],
                padding_mask_videos=av_inputs.get("padding_mask_videos"),
                padding_mask=av_inputs.get("padding_mask"),
            )

            avt_list.append(outputs.text_audio_video_embeds.cpu().numpy())
            at_list.append(outputs.audio_plus_text_embeds.cpu().numpy())
            vt_list.append(outputs.video_plus_text_embeds.cpu().numpy())

            # Raw [CLS] token from ModernBERT last layer (no projection head).
            # This is the upstream text feature that avt / at / vt are all
            # computed from.  PE-AV forces output_hidden_states=True before
            # calling the text model, so hidden_states is always non-None here.
            t_list.append(outputs.text_outputs.hidden_states[-1][:, 0].cpu().numpy())

            i += len(batch)
            print(f"    {i}/{len(pairs)} processed (batch={cur_size})", end="\r")
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            gc.collect()
            if cur_size == 1:
                raise
            cur_size = max(1, cur_size // 2)
            print(f"\n    OOM — reducing batch size to {cur_size}")
    print()
    return {
        "avt": np.concatenate(avt_list, axis=0),
        "at":  np.concatenate(at_list,  axis=0),
        "vt":  np.concatenate(vt_list,  axis=0),
        "t":   np.concatenate(t_list,   axis=0),
    }


In [ ]:
for model_name, cfg in MODEL_CONFIGS.items():
    model_id  = cfg["hf_id"]
    print(f"\n{'='*60}\nModel: {model_id}\n{'='*60}")

    processor = AutoProcessor.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY)
    model     = AutoModel.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY).to(DEVICE)
    model.eval()

    for seg_sec in SEGMENT_DURATIONS:
        out_dir  = OUTPUT_BASE / model_name / f"{int(seg_sec)}s"
        out_dir.mkdir(parents=True, exist_ok=True)

        if all((out_dir / f"{model_name}_{m}.npy").exists() for m in TEXT_MODALITIES):
            print(f"  {seg_sec}s -- all text-modality files exist, skipping")
            continue

        text_file = TEXT_BASE / f"{int(seg_sec)}s" / "rewritten_text_inputs.json"
        if not text_file.exists():
            raise FileNotFoundError(
                f"{text_file}\n"
                f"Run text.ipynb (InternVL3-8B captions + AudioFlamingo3-HF audio captions) "
                f"then rewrite_text_inputs.py before extracting AVT embeddings."
            )

        text_inputs = json.loads(text_file.read_text())
        pairs = find_chunk_pairs(DATA_BASE, seg_sec)

        if len(pairs) != len(text_inputs):
            raise ValueError(
                f"  {seg_sec}s: {len(pairs)} segment pairs but "
                f"{len(text_inputs)} text inputs — re-run text.ipynb"
            )

        print(f"  {seg_sec}s -- extracting from {len(pairs)} pairs ...")
        embeds = extract_avt_embeddings(
            model, processor, pairs, text_inputs, BATCH_SIZE, DEVICE
        )

        for mod, arr in embeds.items():
            out_path = out_dir / f"{model_name}_{mod}.npy"
            np.save(out_path, arr)
            print(f"    saved {out_path.name}  shape={arr.shape}")

    del model, processor
    torch.cuda.empty_cache()
    gc.collect()
    print(f"  {model_name} complete, GPU memory released")

print("\nAll models done.")